In [ ]:
# Agentic Time Series Forecasting with sktime

This notebook demonstrates a simple agentic forecasting system:
- Prompt-based forecasting
- Automatic model selection
- Trend explanation
- Model comparison

In [ ]:
from sktime.datasets import load_airline
from sktime.forecasting.theta import ThetaForecaster
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.base import ForecastingHorizon

In [ ]:
y = load_airline()
y.tail()

In [ ]:
def select_model(prompt):
    prompt = prompt.lower()

    if "smooth" in prompt:
        return ExponentialSmoothing()
    elif "naive" in prompt:
        return NaiveForecaster(strategy="last")
    else:
        return ThetaForecaster(sp=12)

In [ ]:
from sktime.forecasting.model_selection import temporal_train_test_split

prompt = "Predict next 6 months with smooth model"

# split data
y_train, y_test = temporal_train_test_split(y, test_size=6)

model = select_model(prompt)
model.fit(y_train)

fh = ForecastingHorizon(y_test.index, is_relative=False)
preds = model.predict(fh)

preds

In [ ]:
## Model Evaluation

We evaluate the forecasting performance using Mean Absolute Error (MAE).

In [ ]:
from sktime.performance_metrics.forecasting import mean_absolute_error

mae = mean_absolute_error(y_test, preds)
print("MAE:", mae)

In [ ]:
trend = "increasing" if preds.iloc[-1] > y.iloc[-1] else "decreasing or stable"

print(f"Using {model.__class__.__name__}, the forecast shows a {trend} trend.")